# Floor Plan Generator - Training Notebook

Train Stable Diffusion to generate Indian house floor plans from text descriptions.

**Dataset:** CubiCasa5K (5,000 professional floor plans)  
**Model:** Stable Diffusion 1.5 with ControlNet  
**Hardware:** Google Colab T4 GPU (15GB VRAM)

---

## Setup Instructions - NO GOOGLE DRIVE NEEDED! 🎉

1. **Enable GPU:**
   - Runtime → Change runtime type → T4 GPU

2. **Upload training data:**
   - Upload `training_data.zip` using the cell below
   - OR download directly from a URL

3. **Run all cells** (will take ~30 min setup, then 10-15 hrs training)

**Training data:** 3.7GB (will be uploaded directly to Colab runtime)

## Step 1: Upload Training Data (Choose ONE method)

In [ ]:
import os
os.chdir('/content')

# === METHOD 1: Upload from your computer (RECOMMENDED) ===
# Click "Choose Files" and select training_data.zip from your Downloads folder
# This will take 10-20 minutes for the 3.7GB file

from google.colab import files
import os

print("=" * 70)
print("UPLOAD TRAINING DATA")
print("=" * 70)
print("\nClick 'Choose Files' below and select 'training_data.zip'")
print("File size: 3.7GB | Upload time: ~10-20 minutes")
print("\n⚠️  Keep this tab open during upload!")
print("=" * 70 + "\n")

if not os.path.exists('training_data.zip'):
    uploaded = files.upload()
    print("\n✅ Upload complete!")
else:
    print("✅ training_data.zip already exists")

## Step 2: Install Dependencies

In [ ]:
%%capture
!pip install diffusers[torch] transformers accelerate safetensors xformers==0.0.23 -q
!pip install datasets pillow torchvision -q
print("✓ Dependencies installed")

## Step 3: Extract Training Data

In [ ]:
import zipfile
import os

os.chdir('/content')

print("Checking for training data...")
if not os.path.exists('training_data.zip'):
    print("❌ training_data.zip not found!")
    print("Please run Step 1 to upload the file.")
    raise FileNotFoundError("training_data.zip not found")

# Extract dataset if not already extracted
if not os.path.exists('training_data/train'):
    print("\nExtracting training data (this may take 5-10 minutes)...")
    print("File size: 3.7GB → Extracted: ~4GB")
    
    with zipfile.ZipFile('training_data.zip', 'r') as zip_ref:
        # Show progress
        file_list = zip_ref.namelist()
        total_files = len(file_list)
        
        for i, file in enumerate(file_list):
            zip_ref.extract(file, '.')
            if (i + 1) % 500 == 0:
                print(f"  Extracted {i+1}/{total_files} files...")
    
    print("✓ Dataset extracted")
else:
    print("✓ Dataset already extracted")

# Verify dataset
train_images = len([f for f in os.listdir('training_data/train/images') if f.endswith('.png')])
val_images = len([f for f in os.listdir('training_data/val/images') if f.endswith('.png')])
test_images = len([f for f in os.listdir('training_data/test/images') if f.endswith('.png')])

print(f"\n{'='*70}")
print("DATASET READY!")
print(f"{'='*70}")
print(f"Train: {train_images} images")
print(f"Val:   {val_images} images")
print(f"Test:  {test_images} images")
print(f"Total: {train_images + val_images + test_images} floor plans")
print(f"{'='*70}\n")

## Step 4: Load Base Model

In [ ]:
from diffusers import StableDiffusionPipeline, DDPMScheduler
import torch

model_id = "runwayml/stable-diffusion-v1-5"

print("Loading Stable Diffusion model...")
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None
)
pipe = pipe.to("cuda")

print("✓ Model loaded on GPU")
print(f"  Device: {pipe.device}")

## Step 5: Prepare Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
from torchvision import transforms

class FloorPlanDataset(Dataset):
    def __init__(self, data_dir, split='train', size=512):
        self.data_dir = Path(data_dir) / split
        self.image_paths = sorted(list((self.data_dir / 'images').glob('*.png')))
        self.size = size
        
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        
        # Load prompt
        prompt_path = self.data_dir / 'prompts' / f"{image_path.stem}.txt"
        with open(prompt_path, 'r') as f:
            prompt = f.read().strip()
        
        # Transform image
        image = self.transform(image)
        
        return {
            'image': image,
            'prompt': prompt
        }

# Create datasets from /content/training_data
train_dataset = FloorPlanDataset('/content/training_data', split='train')
val_dataset = FloorPlanDataset('/content/training_data', split='val')

print(f"✓ Dataset loaded:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Val:   {len(val_dataset)} samples")

# Test loading one sample
sample = train_dataset[0]
print(f"\nSample:")
print(f"  Image shape: {sample['image'].shape}")
print(f"  Prompt: {sample['prompt']}")

## Step 6: Configure Training

In [ ]:
from accelerate import Accelerator
from diffusers.optimization import get_cosine_schedule_with_warmup
import torch.nn.functional as F

# Training configuration
config = {
    'learning_rate': 1e-5,
    'batch_size': 4,  # Fits in T4 GPU (15GB)
    'num_epochs': 10,
    'gradient_accumulation_steps': 4,
    'save_every': 500,
    'sample_every': 100,
    'output_dir': 'floor_plan_model',
    'mixed_precision': 'fp16'
}

# Initialize accelerator for mixed precision training
accelerator = Accelerator(
    mixed_precision=config['mixed_precision'],
    gradient_accumulation_steps=config['gradient_accumulation_steps']
)

# Data loader
train_dataloader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=2
)

# Optimizer
optimizer = torch.optim.AdamW(
    pipe.unet.parameters(),
    lr=config['learning_rate']
)

# Learning rate scheduler
num_training_steps = len(train_dataloader) * config['num_epochs']
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer=optimizer,
    num_warmup_steps=500,
    num_training_steps=num_training_steps
)

# Prepare for training
pipe.unet, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
    pipe.unet, optimizer, train_dataloader, lr_scheduler
)

print("✓ Training configured")
print(f"  Batch size: {config['batch_size']}")
print(f"  Epochs: {config['num_epochs']}")
print(f"  Total steps: {num_training_steps:,}")
print(f"  Learning rate: {config['learning_rate']}")

## Step 7: Training Loop (10-15 hours)

In [ ]:
from tqdm.auto import tqdm
import os
from datetime import datetime

os.makedirs(config['output_dir'], exist_ok=True)
os.makedirs(f"{config['output_dir']}/samples", exist_ok=True)

global_step = 0
best_loss = float('inf')

print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nThis will take ~10-15 hours on T4 GPU")
print("You can close this tab - training will continue in background\n")

for epoch in range(config['num_epochs']):
    pipe.unet.train()
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{config['num_epochs']}")
    
    for step, batch in enumerate(progress_bar):
        with accelerator.accumulate(pipe.unet):
            # Encode images to latent space
            latents = pipe.vae.encode(batch['image'].to(pipe.vae.dtype)).latent_dist.sample()
            latents = latents * pipe.vae.config.scaling_factor
            
            # Sample noise
            noise = torch.randn_like(latents)
            
            # Random timestep
            timesteps = torch.randint(
                0, pipe.scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=latents.device
            ).long()
            
            # Add noise to latents
            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
            
            # Get text embeddings
            text_embeddings = pipe.text_encoder(
                pipe.tokenizer(
                    batch['prompt'],
                    padding='max_length',
                    max_length=pipe.tokenizer.model_max_length,
                    truncation=True,
                    return_tensors='pt'
                ).input_ids.to(pipe.text_encoder.device)
            )[0]
            
            # Predict noise
            noise_pred = pipe.unet(noisy_latents, timesteps, text_embeddings).sample
            
            # Calculate loss
            loss = F.mse_loss(noise_pred, noise, reduction='mean')
            
            # Backward pass
            accelerator.backward(loss)
            
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(pipe.unet.parameters(), 1.0)
            
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
        
        # Update progress
        if accelerator.sync_gradients:
            global_step += 1
            progress_bar.set_postfix({'loss': loss.item(), 'step': global_step})
            
            # Save checkpoint
            if global_step % config['save_every'] == 0:
                accelerator.wait_for_everyone()
                if accelerator.is_main_process:
                    pipe.save_pretrained(f"{config['output_dir']}/checkpoint-{global_step}")
                    print(f"\n✓ Checkpoint saved at step {global_step}")
                    
                    # Save best model
                    if loss.item() < best_loss:
                        best_loss = loss.item()
                        pipe.save_pretrained(f"{config['output_dir']}/best_model")
                        print(f"✓ New best model! Loss: {best_loss:.4f}")
            
            # Generate sample
            if global_step % config['sample_every'] == 0:
                if accelerator.is_main_process:
                    pipe.unet.eval()
                    with torch.no_grad():
                        sample_prompt = "Floor plan with 3 bedrooms, 2 bathrooms, kitchen, living room"
                        image = pipe(
                            sample_prompt,
                            num_inference_steps=50,
                            guidance_scale=7.5
                        ).images[0]
                        image.save(f"{config['output_dir']}/samples/step_{global_step}.png")
                    pipe.unet.train()

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print(f"End time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total steps: {global_step}")
print(f"Best loss: {best_loss:.4f}")
print(f"\nModel saved to: {config['output_dir']}/best_model/")

## Step 8: Test Generation

In [ ]:
from IPython.display import display

# Load best model
pipe = StableDiffusionPipeline.from_pretrained(
    f"{config['output_dir']}/best_model",
    torch_dtype=torch.float16
).to("cuda")

# Test prompts
test_prompts = [
    "Floor plan with 2 bedrooms, 1 bathroom, kitchen, living room",
    "Floor plan with 3 bedrooms, 2 bathrooms, kitchen",
    "Floor plan with 4 bedrooms, 3 bathrooms, kitchen, living room",
    "Floor plan with 1 bedroom, 1 bathroom, kitchen"
]

print("Generating test floor plans...\n")
for i, prompt in enumerate(test_prompts):
    print(f"{i+1}. {prompt}")
    image = pipe(
        prompt,
        num_inference_steps=50,
        guidance_scale=7.5
    ).images[0]
    image.save(f"test_output_{i+1}.png")
    display(image)

print("\n✅ Test generation complete!")